Notebook 1. Proyección ortogonal sobre $\mathcal{L}(\boldsymbol{1})$ en $\mathbb{R}^3$
======================================================================================

**Author:** Marcos Bujosa



\maketitle

<div class="abstract" id="orga2be4b3">
<p>
Visualización interactiva en $\mathbb{R}^3$ de la proyección ortogonal sobre $\mathcal{L}(\boldsymbol{1})$ (la recta de los vectores constantes): la media aritmética, la desviación típica y, con dos vectores, el ángulo entre sus vectores en desviaciones (la correlación). Complemento de las lecciones 4 y 5.
</p>

</div>

-   ([mybinder](https://mybinder.org/v2/gh/mbujosab/PEconometria/gh-pages?labpath=CuadernosElectronicos/S05-Notebook01.ipynb))



## Introducción



Este cuaderno acompaña a las lecciones 4 y 5. En ellas vimos que:

-   la *media aritmética* $\mu_{\boldsymbol{y}}$ es el escalar que multiplica a $\boldsymbol{1}$ para obtener la *proyección ortogonal* de $\boldsymbol{y}$ sobre la recta $\mathcal{L}(\boldsymbol{1})$;
-   la *desviación típica* $\sigma_{\boldsymbol{y}}$ es la norma del *vector en desviaciones* $\boldsymbol{y}-\boldsymbol{\mathop{\overline{y}}}$, perpendicular a $\boldsymbol{1}$;
-   con *dos* vectores $\boldsymbol{v}$ y $\boldsymbol{x}$, la *correlación* $\rho_{\boldsymbol{v}\boldsymbol{x}}$ es el coseno del ángulo entre sus respectivos vectores en desviaciones.

Además, sobre la recta azul y sobre los segmentos en desviaciones verás
pequeñas cruces: son una regla graduada. Pasa el ratón por encima de
cualquiera de ellas para ver el valor exacto que representa ($\mu$ o
$\sigma$, según el caso), y cuéntalas desde el origen (o desde la
proyección) para leerlos directamente en la figura, exactamente igual
que contabas diagonales en las cuadrículas de papel de la lección 4.

Todas las figuras de las lecciones 4 y 5 son necesariamente esquemáticas ($\mathbb{R}^2$ o $\mathbb{R}^3$ dibujados en el papel). Aquí podrás *rotar* las figuras de verdad con el ratón (arrastrando sobre ellas) y *cambiar los vectores* editando una celda y volviéndola a ejecutar.

Trabajamos en $\mathbb{R}^3$, es decir, con *tres* observaciones ($n=3$).



## Herramientas auxiliares



#### Módulos y configuración



Cargamos `numpy` para el álgebra vectorial y `plotly` para las figuras 3D interactivas.



In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Para que las figuras se incrusten completas en el notebook (sin depender
# de conexión a internet ni de extensiones de JupyterLab):
pio.renderers.default = "notebook"

#### Media, vector de medias y vector en desviaciones



Estas funciones traducen a código las definiciones de la lección 4: el
*vector de medias* $\boldsymbol{\mathop{\overline{v}}}=\mu_{\boldsymbol{v}}\boldsymbol{1}$ (la proyección de
$\boldsymbol{v}$ sobre $\mathcal{L}(\boldsymbol{1})$) y el *vector en desviaciones*
$\boldsymbol{v}-\boldsymbol{\mathop{\overline{v}}}$. Añadimos también, ya de paso, la *covarianza* y la
*correlación* de la lección 5 (producto escalar y coseno del ángulo entre
dos vectores en desviaciones), que necesitaremos en la Parte 2.



In [1]:
def vector_de_medias(v):
    """Devuelve (vector_de_medias, mu): el vector constante mu*1 y el
    escalar mu = media aritmética de las componentes de v."""
    v = np.asarray(v, dtype=float)
    mu = v.mean()
    return np.full(v.shape, mu), mu

def vector_en_desviaciones(v):
    """Devuelve (v - vector_de_medias(v), mu)."""
    v = np.asarray(v, dtype=float)
    vbar, mu = vector_de_medias(v)
    return v - vbar, mu

def covarianza(a, b):
    """Producto escalar (estadístico) de los vectores en desviaciones de a y b."""
    ea, _ = vector_en_desviaciones(a)
    eb, _ = vector_en_desviaciones(b)
    return np.mean(ea * eb)

def correlacion(a, b):
    """Coseno del ángulo entre los vectores en desviaciones de a y b."""
    return covarianza(a, b) / np.sqrt(covarianza(a, a) * covarianza(b, b))

#### Una base ortonormal del plano $\mathcal{L}(\boldsymbol{1})^\perp$



Para poder dibujar una rejilla *dentro* del propio plano perpendicular a $\boldsymbol{1}$ (y así poder \`\`contar'' allí normas y ángulos, como en el papel cuadriculado de las figuras 2D de las lecciones), necesitamos dos vectores unitarios y ortogonales entre sí, ambos perpendiculares a $\boldsymbol{1}$.



In [1]:
def base_plano_ortogonal():
    """Base ortonormal (u1, u2) del plano L(1)^perp en R^3."""
    uno = np.array([1., 1., 1.]) / np.sqrt(3)
    u1 = np.array([1., -1., 0.])
    u1 = u1 / np.linalg.norm(u1)          # ya es perpendicular a (1,1,1)
    u2 = np.cross(uno, u1)                # perpendicular a ambos
    u2 = u2 / np.linalg.norm(u2)
    return u1, u2

#### Funciones de dibujo



Una *decisión de diseño*: en lugar de rellenar el cubo con cubitos
semitransparentes de lado 1 (en 3D esto se satura enseguida y resulta
ilegible), usamos:

-   la rejilla *nativa* de Plotly en las tres paredes de fondo del cubo de
    ejes (parámetro `dtick=1`), como sustituto del papel cuadriculado 2D;
-   una rejilla propia, unitaria, *dentro* del plano $\mathcal{L}(\boldsymbol{1})^\perp$,
    construida con la base ortonormal anterior;
-   opcionalmente (activable más abajo), los puntos de coordenadas enteras
    del cubo, para poder verificar longitudes contando saltos.

Sin embargo, esta rejilla de fondo *no* permite leer $\mu$ ni $\sigma$
contando cuadritos directamente, por una razón geométrica: un cubo de
lado 1 tiene diagonal $\sqrt3$ (Pitágoras, aplicado dos veces), de modo
que un incremento de una unidad en la *media* —que se desplaza a lo
largo de la diagonal $(1,1,1)$— corresponde a una distancia euclídea de
$\sqrt3$, no de $1$. Es la misma lógica que en las figuras de papel de la
lección 4, donde una diagonal del cuadrado (que mide $\sqrt2$, no $1$)
equivale a un incremento de una unidad en $\mu$.

Para poder contar de verdad, añadimos dos reglas graduadas propias, con
marcas en forma de cruz:

-   `marcas_unitarias_L1`: sobre la recta $\mathcal{L}(\boldsymbol{1})$, situadas en los
    puntos $k\cdot(1,1,1)$ para $k$ entero; cada cruz sucesiva representa
    un incremento de exactamente una unidad en $\mu$ (sirve para cualquier
    vector que se proyecte, no solo para uno en particular).
-   `marcas_desviacion`: sobre cada vector en desviaciones dibujado (tanto
    en su posición original, junto a la proyección, como trasladado al
    origen), espaciadas para que cada cruz sucesiva represente un
    incremento de una unidad en $\sigma$.

Pasa el ratón sobre cualquier cruz para ver su valor exacto.



In [1]:
def marcas_unitarias_L1(fig, rango=4, color='blue'):
    """Marcas sobre la recta L(1) situadas en los puntos k*(1,1,1) para
    k entero. Cada marca corresponde EXACTAMENTE a un valor entero de la
    media mu (el escalar que multiplica a 1 para obtener la proyección
    y_barra = mu*1). Por eso basta con contar marcas desde el origen
    hasta la proyección para leer mu_y directamente (con signo).
    Nota: la distancia EUCLÍDEA entre dos marcas consecutivas es
    ||(1,1,1)||_e = sqrt(3), la diagonal de un cubo de lado 1 (el
    equivalente 3D de la diagonal de un cuadrado, sqrt(2), en las
    figuras 2D de las lecciones)."""
    uno = np.array([1., 1., 1.])
    ks = np.arange(-rango, rango + 1)
    puntos = np.outer(ks, uno)
    fig.add_trace(go.Scatter3d(
        x=puntos[:, 0], y=puntos[:, 1], z=puntos[:, 2],
        mode='markers',
        marker=dict(size=4, color=color, symbol='cross'),
        text=[f"mu = {k}" for k in ks],
        hoverinfo='text',
        showlegend=False))

def marcas_desviacion(fig, base, e, n, color, etiqueta='sigma'):
    """Marcas sobre el segmento que va desde 'base' en la dirección del
    vector en desviaciones 'e', espaciadas de modo que cada marca sucesiva
    corresponde a un incremento de 1 en sigma (la norma ESTADÍSTICA de e).
    Solo se dibujan las marcas que caben dentro del segmento, es decir,
    k = 0, 1, ..., floor(sigma) (nunca se sobrepasa la punta del vector).

    Misma lógica que las marcas de L(1): una unidad de sigma corresponde a
    una distancia EUCLÍDEA de sqrt(n) a lo largo de la dirección de e."""
    norma_euclidea_e = np.linalg.norm(e)
    if norma_euclidea_e < 1e-12:
        return  # vector en desviaciones nulo: no hay nada que marcar
    sigma = norma_euclidea_e / np.sqrt(n)
    u_e = e / norma_euclidea_e
    paso_euclideo = np.sqrt(n)          # distancia euclídea <-> Δsigma = 1
    kmax = int(np.floor(sigma + 1e-9))  # tolerancia para el caso sigma entero
    ks = np.arange(0, kmax + 1)
    puntos = base + np.outer(ks, u_e * paso_euclideo)
    fig.add_trace(go.Scatter3d(
        x=puntos[:, 0], y=puntos[:, 1], z=puntos[:, 2],
        mode='markers',
        marker=dict(size=4, color=color, symbol='cross'),
        text=[f"{etiqueta} = {k}" for k in ks],
        hoverinfo='text',
        showlegend=False))    

def recta_L1(fig, rango=4, color='blue'):
    """Dibuja la recta L(1) = múltiplos de (1,1,1), con marcas de regla
    que representan incrementos unitarios de la media mu (ver
    marcas_unitarias_L1; la distancia EUCLÍDEA entre dos marcas
    consecutivas es sqrt(3), no 1)."""
    t = np.linspace(-rango, rango, 2)
    fig.add_trace(go.Scatter3d(
        x=t, y=t, z=t, mode='lines',
        line=dict(color=color, width=5),
        name='L(1)'))
    marcas_unitarias_L1(fig, rango=rango, color=color)

def plano_ortogonal(fig, rango=4, color='lightslategray'):
    """Dibuja un parche semitransparente del plano L(1)^perp, más una
    rejilla unitaria dentro de él (usando su propia base ortonormal)."""
    u1, u2 = base_plano_ortogonal()

    # parche semitransparente (solo para "ver" el plano)
    S, T = np.meshgrid([-rango, rango], [-rango, rango])
    X = S * u1[0] + T * u2[0]
    Y = S * u1[1] + T * u2[1]
    Z = S * u1[2] + T * u2[2]
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z, showscale=False, opacity=0.12,
        colorscale=[[0, color], [1, color]], hoverinfo='skip'))

    # rejilla unitaria dentro del plano
    for s in range(-rango, rango + 1):
        p0 = s * u1 - rango * u2
        p1 = s * u1 + rango * u2
        fig.add_trace(go.Scatter3d(
            x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
            mode='lines', line=dict(color='gray', width=1),
            showlegend=False, hoverinfo='skip'))
    for t in range(-rango, rango + 1):
        p0 = -rango * u1 + t * u2
        p1 = rango * u1 + t * u2
        fig.add_trace(go.Scatter3d(
            x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
            mode='lines', line=dict(color='gray', width=1),
            showlegend=False, hoverinfo='skip'))

def agrega_vector(fig, punto, color='crimson', nombre='v', dash='solid'):
    """Dibuja el vector desde el origen hasta 'punto' (línea + marcador
    en la punta, a modo de flecha)."""
    punto = np.asarray(punto, dtype=float)
    fig.add_trace(go.Scatter3d(
        x=[0, punto[0]], y=[0, punto[1]], z=[0, punto[2]],
        mode='lines+markers',
        line=dict(color=color, width=6, dash=dash),
        marker=dict(size=[0, 5], color=color),
        name=nombre))

def agrega_segmento(fig, p0, p1, color='gray'):
    """Segmento discontinuo entre dos puntos (el 'cateto' de Pitágoras
    que conecta la punta de la media con la punta del vector)."""
    fig.add_trace(go.Scatter3d(
        x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
        mode='lines', line=dict(color=color, width=3, dash='dot'),
        showlegend=False, hoverinfo='skip'))

def agrega_reticula_puntos(fig, rango=4, color='lightslategray'):
    """Puntos de coordenadas enteras dentro del cubo (para poder 'contar'
    a lo largo de cualquier dirección). Desactivado por defecto."""
    coords = range(-rango, rango + 1)
    X, Y, Z = np.meshgrid(coords, coords, coords, indexing='ij')
    fig.add_trace(go.Scatter3d(
        x=X.ravel(), y=Y.ravel(), z=Z.ravel(),
        mode='markers',
        marker=dict(size=2, color=color, opacity=0.3),
        showlegend=False, hoverinfo='skip'))

def figura_R3(vectores, colores, nombres, rango=4,
              mostrar_desviaciones_en_origen=False,
              mostrar_reticula_puntos=False,
              titulo=""):
    """Compone la escena completa: L(1), el plano L(1)^perp, y para cada
    vector de la lista: el vector, su vector de medias (proyección),
    el segmento que los conecta y, opcionalmente, su vector en
    desviaciones trasladado al origen (para comparar ángulos)."""
    fig = go.Figure()
    recta_L1(fig, rango=rango)
    plano_ortogonal(fig, rango=rango)
    if mostrar_reticula_puntos:
        agrega_reticula_puntos(fig, rango=rango)

    for v, color, nombre in zip(vectores, colores, nombres):
        v = np.asarray(v, dtype=float)
        vbar, mu = vector_de_medias(v)
        e = v - vbar
        n = len(v)

        agrega_vector(fig, v, color=color, nombre=nombre)
        agrega_vector(fig, vbar, color='blue', nombre=f'media de {nombre}')
        agrega_segmento(fig, vbar, v, color=color)
        marcas_desviacion(fig, vbar, e, n, color=color,
                           etiqueta=f'sigma_{nombre}')

        if mostrar_desviaciones_en_origen:
            agrega_vector(fig, e, color=color, dash='dash',
                          nombre=f'{nombre} - media (trasladado al origen)')
            marcas_desviacion(fig, np.zeros(3), e, n, color=color,
                               etiqueta=f'sigma_{nombre}')

    fondo = dict(showbackground=True, backgroundcolor='rgb(235,235,245)',
                 gridcolor='white', dtick=1, range=[-rango, rango],
                 #zeroline=True, zerolinecolor='black', zerolinewidth=1
                 )
    fig.update_layout(
        scene=dict(xaxis=fondo, yaxis=fondo, zaxis=fondo,
                    aspectmode='cube'),
        title=titulo, width=800, height=700,
        margin=dict(l=0, r=0, b=0, t=40))
    return fig

## Parte 1: un vector — la media como proyección



*(Cobra la geometría de la lección 4.)*

Edita las componentes de $\boldsymbol{y}$ y vuelve a ejecutar la celda para ver una figura nueva.



In [1]:
y = np.array([1., 2., 3.])

In [1]:
fig = figura_R3([y], ['crimson'], ['y'], rango=4,
                 titulo="Proyección de y sobre L(1)")
fig.show()

Arrastra sobre la figura para rotarla. Observa:

-   la recta azul $\mathcal{L}(\boldsymbol{1})$;
-   el vector rojo $\boldsymbol{y}$ y su proyección azul $\boldsymbol{\mathop{\overline{y}}}$ sobre esa recta;
-   el segmento punteado que une $\boldsymbol{\mathop{\overline{y}}}$ con $\boldsymbol{y}$: es el
    *vector en desviaciones* $\boldsymbol{y}-\boldsymbol{\mathop{\overline{y}}}$, y por construcción es
    perpendicular a la recta azul (atención: *no* es perpendicular al
    plano gris, sino que está *contenido* en él —ese plano gris es
    precisamente $\mathcal{L}(\boldsymbol{1})^\perp$ trasladado al punto
    $\boldsymbol{\mathop{\overline{y}}}$);
-   las pequeñas cruces sobre la recta azul y sobre el segmento punteado:
    cuéntalas desde el origen hasta $\boldsymbol{\mathop{\overline{y}}}$ para leer
    $\mu_{\boldsymbol{y}}$ (con signo), y desde $\boldsymbol{\mathop{\overline{y}}}$ hasta $\boldsymbol{y}$
    para leer $\sigma_{\boldsymbol{y}}$; pasa el ratón por encima de cualquiera
    para ver su valor exacto.



#### Comprobación numérica: Pitágoras



Una advertencia de escala antes de medir nada *a ojo* en la figura: la
*posición* de los puntos y el *ángulo* que se aprecia son exactamente los
de las lecciones, con independencia de cómo se normalice el producto
escalar. Pero la *longitud euclídea* del segmento punteado, medida con
una regla ingenua, no es literalmente $\sigma_{\boldsymbol{y}}$ (la norma
**estadística**, con el factor $1/n$), sino $\sigma_{\boldsymbol{y}}\sqrt{n}$.

Por suerte no hace falta hacer esa corrección de cabeza: las cruces
dibujadas sobre el propio segmento ya la incorporan. Cada cruz sucesiva
corresponde exactamente a un incremento de una unidad en
$\sigma_{\boldsymbol{y}}$ (compruébalo pasando el ratón por encima). La celda
siguiente solo confirma numéricamente lo que ya puedes leer contando
cruces en la figura.



In [1]:
n = len(y)
e, mu_y = vector_en_desviaciones(y)

sigma_y = np.sqrt(np.mean(e**2))          # norma ESTADÍSTICA (con 1/n)
norma_euclidea_segmento = np.linalg.norm(e)  # lo que "mide" la regla sobre la figura

print(f"mu_y = {mu_y}")
print(f"sigma_y (norma estadística) = {sigma_y:.4f}")
print(f"norma euclídea del segmento dibujado = {norma_euclidea_segmento:.4f}")
print(f"sigma_y * sqrt(n) = {sigma_y*np.sqrt(n):.4f}  (debe coincidir con la anterior)")

# Teorema de Pitágoras (estadístico): ||y||_s^2 = mu_y^2 + sigma_y^2
print()
print(f"media(y**2) = {np.mean(y**2):.4f}")
print(f"mu_y**2 + sigma_y**2 = {mu_y**2 + sigma_y**2:.4f}")

#### Actividad



Cambia $\boldsymbol{y}$ (por ejemplo, a un vector con media negativa, como $(2,-1,-7)$) y vuelve a ejecutar las dos celdas anteriores. Observa cómo cambia la posición de $\boldsymbol{\mathop{\overline{y}}}$ sobre la recta azul (¿a qué lado del origen queda cuando $\mu_{\boldsymbol{y}}<0$?), y comprueba que la identidad de Pitágoras se sigue cumpliendo.



## Parte 2: dos vectores — el ángulo y la correlación



*(Cobra la geometría de la lección 5.)*

Retomamos el ejemplo numérico de la lección 5: $\boldsymbol{x}=(1,2,3)$ y $\boldsymbol{v}=(2,2,5)$, con $\rho_{\boldsymbol{v}\boldsymbol{x}}=\frac{\sqrt3}{2}$ (un ángulo de $30^o$: ni ortogonales, ni alineados).



In [1]:
x = np.array([1., 2., 3.])
v = np.array([2., 2., 5.])

In [1]:
fig2 = figura_R3([v, x], ['crimson', 'seagreen'], ['v', 'x'], rango=5,
                  mostrar_desviaciones_en_origen=True,
                  titulo="v y x: proyecciones y vectores en desviaciones")
fig2.show()

Además de lo visto en la Parte 1 (para cada vector, por separado: $\boldsymbol{v}$
en rojo/crimson, $\boldsymbol{x}$ en verde/seagreen), ahora se dibujan también,
con línea discontinua y *trasladados al origen*, los dos vectores en
desviaciones $\boldsymbol{v}-\boldsymbol{\mathop{\overline{v}}}$ y $\boldsymbol{x}-\boldsymbol{\mathop{\overline{x}}}$,
cada uno con sus propias cruces de regla (una por unidad de $\sigma$).
Rota la figura hasta colocar el punto de vista casi sobre el plano gris
(mirándolo \`\`de frente''): así se aprecia mejor el ángulo $\theta$ entre
ambos, tal como en la segunda imagen de la sección \`\`Ilustración
geométrica de la correlación'' de la lección 5. De paso, comparando
cuántas cruces caben en cada vector trasladado, puedes leer también, a
ojo, cuál de los dos tiene mayor $\sigma$.



#### Comprobación numérica: covarianza, correlación y ángulo



Verificamos que la covarianza y la correlación calculadas algebraicamente coinciden con el ángulo que se aprecia geométricamente entre los vectores en desviaciones dibujados en la figura anterior.



In [1]:
sigma_vx = covarianza(v, x)
rho_vx = correlacion(v, x)

# el mismo ángulo, calculado directamente sobre los vectores en desviaciones
# dibujados en la figura (con el producto escalar euclídeo; el factor 1/n
# se cancela en el cociente, así que el coseno coincide con rho_vx)
ev, _ = vector_en_desviaciones(v)
ex, _ = vector_en_desviaciones(x)
cos_theta = np.dot(ev, ex) / (np.linalg.norm(ev) * np.linalg.norm(ex))
theta_grados = np.degrees(np.arccos(cos_theta))

print(f"sigma_vx (covarianza) = {sigma_vx:.4f}")
print(f"rho_vx (correlación)  = {rho_vx:.4f}")
print(f"cos(theta) calculado sobre la figura = {cos_theta:.4f}")
print(f"theta = {theta_grados:.2f} grados")

#### Actividad



Modifica $\boldsymbol{v}$ y $\boldsymbol{x}$ y vuelve a ejecutar las celdas anteriores para explorar los casos extremos:

-   Vectores en desviaciones *ortogonales* ($\rho=0$, $\theta=90^o$): por ejemplo, prueba con $\boldsymbol{x}=(1,2,3)$ y $\boldsymbol{v}=(1,3,2)$.
-   Vectores en desviaciones *alineados* ($\rho=\pm1$): por ejemplo, $\boldsymbol{x}=(1,2,3)$ y $\boldsymbol{v}=2\boldsymbol{x}$ (¿y si tomas $\boldsymbol{v}=-2\boldsymbol{x}$?).

En ambos casos, comprueba en la figura que el resultado coincide con lo que predice la geometría: en el primer caso, los vectores discontinuos trasladados al origen deben verse perpendiculares; en el segundo, deben verse sobre la misma recta (misma dirección, o dirección opuesta).



## Para seguir explorando



-   El parámetro `rango` de `figura_R3` controla el tamaño de la escena (auméntalo si tus vectores tienen componentes grandes).
-   Puedes activar la rejilla de puntos de coordenadas enteras con `mostrar_reticula_puntos=True` al llamar a `figura_R3`; por defecto está desactivada porque con muchos puntos la figura se recarga visualmente.
-   Vuelve a la lección 4 para releer la definición de $\boldsymbol{\mathop{\overline{y}}}$ y $\sigma_{\boldsymbol{y}}$, y a la lección 5 para la covarianza y la correlación como coseno de un ángulo.

